# IAN 601 — Module 10: Pandas for Real Analytics

Pandas is one of the most widely used Python libraries for working with tabular data.

A Pandas **DataFrame** represents rows and columns similar to a spreadsheet or database table, while still allowing us to use Python for filtering, cleaning, transformation, aggregation, and validation.

This notebook is designed to stand alone and uses one sales dataset throughout.

**Workflow:** load → inspect → select → clean → transform → summarize → combine → validate

## Learning goals

By the end of this notebook, you should be able to:

1. load CSV data into a DataFrame;
2. inspect rows, columns, shape, and data types;
3. select columns with bracket notation;
4. select rows and columns with `loc` and `iloc`;
5. filter and sort records;
6. identify missing values and duplicates;
7. convert text, numeric, and date columns safely;
8. create derived columns;
9. summarize data with `groupby()` and `agg()`;
10. create pivot tables;
11. combine tables with `merge()` and `concat()`;
12. validate the cleaned result before reporting it.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

## 1. Import Pandas

Pandas is usually imported with the alias `pd`.

In [ ]:
import pandas as pd

pd.__version__

## 2. Load the Module 10 dataset

When you open this notebook in Colab, read the CSV directly from GitHub.

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module10_Module_Sales_300.csv"

orders = pd.read_csv(data_url)
orders.head()

If you download the notebook and CSV to the same folder on your computer, you can instead use:

```python
orders = pd.read_csv("IAN601_Module10_Module_Sales_300.csv")
```

## 3. First inspection

Before cleaning anything, check what actually loaded.

In [ ]:
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())

orders.head(8)

In [ ]:
orders.sample(5, random_state=10)

## 4. Data types and structure

The `amount` and `order_date` columns are intentionally not fully clean yet.

In [ ]:
orders.dtypes

In [ ]:
orders.info()

## 5. Select columns

A single column returns a Series. A list of columns returns a DataFrame.

In [ ]:
orders["amount"].head()

In [ ]:
orders[["order_id", "channel", "amount"]].head()

## 6. Use loc and iloc

- `loc` selects by labels or conditions.
- `iloc` selects by integer position.

In [ ]:
orders.loc[0:4, ["order_id", "channel", "amount"]]

In [ ]:
orders.iloc[0:5, 0:4]

## 7. Filter and sort rows

For now, filter using columns that are already numeric or clean enough to use.

In [ ]:
web_orders = orders[orders["channel"].str.strip().str.lower() == "web"]

web_orders[["order_id", "channel", "quantity"]].head()

In [ ]:
orders.sort_values("quantity", ascending=False).head(10)

## 8. Profile data quality

Check missing values and duplicate rows before making changes.

In [ ]:
orders.isna().sum()

In [ ]:
print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)].sort_values("order_id")

## 9. Create a clean working copy

Keep the original DataFrame available so you can compare before and after.

In [ ]:
clean = orders.copy()

### Clean text columns

In [ ]:
clean["channel"] = clean["channel"].str.strip().str.lower()
clean["status"] = clean["status"].str.strip().str.lower()

clean[["channel", "status"]].head(10)

### Convert amount to numeric

The CSV stores amounts with a dollar sign. Remove it first, then convert.

In [ ]:
clean["amount"] = pd.to_numeric(
    clean["amount"].str.replace("$", "", regex=False),
    errors="coerce"
)

clean["amount"].head(10)

### Convert dates safely

Invalid dates become missing date values when `errors="coerce"` is used.

In [ ]:
clean["order_date"] = pd.to_datetime(
    clean["order_date"],
    errors="coerce"
)

clean[["order_id", "order_date"]].head()

### Handle missing values

The choice depends on the meaning of the column. Here we use simple teaching examples.

In [ ]:
clean["city"] = clean["city"].fillna("Unknown")
clean["region"] = clean["region"].fillna("Unknown")
clean["amount"] = clean["amount"].fillna(clean["amount"].median())

clean.isna().sum()

### Remove exact duplicate rows

In [ ]:
print("Rows before:", len(clean))

clean = clean.drop_duplicates().copy()

print("Rows after:", len(clean))

## 10. Create derived columns

In [ ]:
clean["tax"] = clean["amount"] * 0.07
clean["total_with_tax"] = clean["amount"] + clean["tax"]

clean[["order_id", "amount", "tax", "total_with_tax"]].head().round(2)

In [ ]:
clean["month"] = clean["order_date"].dt.month_name()
clean["weekday"] = clean["order_date"].dt.day_name()
clean["quarter"] = clean["order_date"].dt.quarter

clean[["order_date", "month", "weekday", "quarter"]].head()

## 11. GroupBy and aggregation

Now that the columns are cleaner, answer questions by category.

In [ ]:
channel_summary = (
    clean.groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum"),
        avg_order=("amount", "mean")
    )
)

channel_summary.round(2)

In [ ]:
region_summary = (
    clean.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum")
    )
    .sort_values("revenue", ascending=False)
)

region_summary.round(2)

## 12. Pivot table

A pivot table summarizes values across two categorical dimensions.

In [ ]:
pivot = pd.pivot_table(
    clean,
    values="amount",
    index="region",
    columns="channel",
    aggfunc="sum",
    fill_value=0
)

pivot.round(2)

## 13. Merge related tables

Create a small customer lookup table and merge it with the order data.

In [ ]:
customer_ids = sorted(clean["customer_id"].dropna().unique())

customers = pd.DataFrame({
    "customer_id": customer_ids,
    "segment": [
        ["consumer", "business", "education"][i % 3]
        for i in range(len(customer_ids))
    ]
})

customers.head()

In [ ]:
merged = clean.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

merged[["order_id", "customer_id", "segment", "amount"]].head()

## 14. Concatenate similar tables

Concatenation stacks tables with similar columns.

In [ ]:
january = clean[clean["order_date"].dt.month == 1].head(5)
february = clean[clean["order_date"].dt.month == 2].head(5)

sample_two_months = pd.concat(
    [january, february],
    ignore_index=True
)

sample_two_months[["order_id", "order_date", "channel", "amount"]]

## 15. Validate the cleaned data

A workflow is not finished until you check the result.

In [ ]:
print("Rows:", len(clean))
print("Duplicate rows:", clean.duplicated().sum())
print("Duplicate order IDs:", clean["order_id"].duplicated().sum())
print("Missing amount values:", clean["amount"].isna().sum())
print("Invalid/missing dates:", clean["order_date"].isna().sum())
print("Total revenue:", round(clean["amount"].sum(), 2))

## 16. Compact analyst workflow

The steps below are the main pattern to remember:

1. Load
2. Inspect
3. Clean
4. Transform
5. Summarize
6. Validate

In [ ]:
analysis = (
    clean[clean["status"] == "complete"]
    .groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum"),
        avg_order=("amount", "mean")
    )
    .sort_values("revenue", ascending=False)
)

analysis.round(2)

## Practice

Try these without copying an earlier cell:

1. Find the five largest completed orders.
2. Calculate total revenue by category.
3. Create a pivot table with category as rows and channel as columns.
4. Find the number of orders in each month.
5. Check whether every customer ID in the order table matched a customer segment.

In [ ]:
# 1. Five largest completed orders


# 2. Revenue by category


# 3. Category x channel pivot table


# 4. Orders by month


# 5. Check the merge


## Summary and self-check

A reliable Pandas workflow is more important than memorizing dozens of methods.

Make sure you can explain:

- the difference between a Series and a DataFrame;
- what `head()`, `shape`, `dtypes`, and `info()` tell you;
- when to use `loc` versus `iloc`;
- why type conversion should be checked carefully;
- what `groupby()` does;
- how a pivot table summarizes categories;
- the difference between `merge()` and `concat()`;
- why row counts, missingness, duplicates, and totals should be checked after cleaning.

Use the practice section to rebuild part of the workflow without copying earlier cells.